# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

- Gymnasium reemplaza por completo a la clase `CheeseWorld`: el objeto `env` expone el mismo contrato `(s,a) -> (s',r,done)` mediante `env.reset()` y `env.step(action)`, además de `env.observation_space.n` y `env.action_space.n` para saber cuántos estados y acciones existen, sin que nosotros tengamos que programar la dinámica del mundo.
- El agente ya no es una clase: quedó **implícito y repartido** entre la Q-table (`Q`, un simple arreglo de NumPy) y las funciones sueltas `epsilon_greedy_policy` y `train_q_learning`, que juegan el mismo papel que los métodos `choose_action` y `update_q` de `QLearningAgent` en CheeseWorld.
- La lógica de la actualización de Q (TD target, TD error, α, γ) es idéntica conceptualmente a CheeseWorld; lo que cambia es la forma de organizar el código: de programación orientada a objetos a funciones que operan sobre un arreglo `Q` y un `env` estándar.


## 2. Crear FrozenLake


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

- **Filas:** 16, una por cada estado (`env.observation_space.n`). FrozenLake 4x4 tiene 16 celdas numeradas de 0 a 15 en orden fila-mayor (0-3 primera fila, 4-7 segunda, etc.).
- **Columnas:** 4, una por cada acción (`env.action_space.n`): Left=0, Down=1, Right=2, Up=3.
- **`Q[s,a]`** representa el **retorno esperado descontado** si el agente está en el estado `s`, ejecuta la acción `a`, y de ahí en adelante sigue la mejor política posible. A diferencia de CheeseWorld (donde `Q` era un diccionario con solo las acciones válidas por estado), aquí la Q-table es una matriz densa `(16, 4)`: existen entradas incluso para acciones que chocan contra un borde, ya que en ese caso el agente simplemente permanece en el mismo estado.


In [3]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- `Q(s,a)=0` para todo par no significa que las acciones sean malas: significa que el agente **no tiene ninguna información todavía**. Es un punto de partida neutral, no una evaluación negativa. Al principio, todos los pares estado-acción son igual de "atractivos" (todos valen 0), así que la política greedy inicial es arbitraria.
- Cuando varias acciones empatan en el máximo (lo cual ocurre todo el tiempo al inicio, ya que todas valen 0), el código desempata **aleatoriamente** con `np.flatnonzero(Qtable[state] == max_q)` seguido de `np.random.choice`. Esto es exactamente el mismo mecanismo de desempate que usábamos en CheeseWorld con `random.choice(best_actions)` sobre el diccionario `Q[state]`.


## 3. Ejecutar una transición


In [4]:
state, _ = env.reset(seed=0)
env.action_space.seed(0)

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 3
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(s=0,\ a=3\ (\text{Up}),\ r=0,\ s'=0)
$$

En este caso el agente estaba en el estado inicial (`s=0`), la política totalmente exploratoria (`epsilon=1.0`) eligió la acción `Up`, pero como `Up` desde la fila superior sale del grid, el ambiente lo dejó en el mismo estado (`s'=0`) y no dio recompensa (`r=0`, `done=False`).

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?

> No teníamos el **`s'` (ni la recompensa `r` asociada) como una distribución completa**: en Value Iteration conocíamos de antemano `T(s,a,s')` para *todas* las transiciones posibles antes de actuar. Aquí, en cambio, solo obtenemos **una muestra realizada** de esa transición cada vez que llamamos a `env.step(action)` — no conocemos la probabilidad de las otras transiciones posibles, solo observamos el resultado de la que efectivamente ocurrió. Q-Learning aprende a partir de esas muestras (experiencia), no del modelo completo.


## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [5]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [6]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


error: XDG_RUNTIME_DIR is invalid or not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [7]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [8]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.

> Lo único que cambió fue el **contenido de la Q-table** — el ambiente, los estados y las acciones disponibles son exactamente los mismos en los tres momentos. Con `Q_initial` (todo en ceros) el agente actuó completamente al azar (recompensa total = 0.0: cayó en un hoyo o no llegó a la meta). Con `Q_early` (solo 50 episodios) la Q-table ya tiene algunos valores distintos de cero, pero todavía no lo suficientemente informativos como para evitar los hoyos de forma confiable (recompensa total = 0.0 también en esta corrida). Con `Q_trained` (10,000 episodios) la Q-table converge a valores que reflejan correctamente qué acciones llevan hacia la meta evitando los hoyos, y el agente llega exitosamente (recompensa total = 1.0). El comportamiento del agente en cada momento es puramente un reflejo de qué tan bien la Q-table aproxima el retorno esperado real de cada par estado-acción.


### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** `s = 14` (la celda justo antes de la meta, adyacente a `s = 15`)

**Valores Q:**

- Left: 0.902
- Down: 0.950
- Right: 1.000
- Up: 0.902

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

> `Right`, con `Q(14, Right) = 1.000`.

**Interpretación:**

> El estado 14 está inmediatamente a la izquierda de la meta (estado 15). Tiene sentido que `Right` tenga el valor más alto y muy cercano a la recompensa de meta (+1): moverse a la derecha desde ahí termina el episodio con éxito de inmediato, así que casi no hay descuento de por medio (`γ` apenas se aplica una vez). Las otras tres acciones (`Left`, `Down`, `Up`) tienen valores un poco menores porque, aunque también pueden eventualmente llevar a la meta, requieren más pasos y por lo tanto más descuento por `γ=0.95`.


## 5. Evaluar la política aprendida


In [9]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> La exploración (elegir acciones al azar con probabilidad ε) tiene un único propósito: permitir que el agente **descubra** información nueva sobre el ambiente mientras todavía está aprendiendo la Q-table. Una vez que el entrenamiento terminó y queremos **medir qué tan buena es la política aprendida**, explorar solo introduciría ruido innecesario y subestimaría el desempeño real del agente — estaríamos penalizando a la política por acciones aleatorias que ella nunca habría elegido. Por eso en evaluación usamos la política puramente greedy: así medimos el comportamiento que el agente realmente ejecutaría en producción, aprovechando (explotando) todo lo que aprendió.


## 6. Experimento: FrozenLake estocástico


In [10]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.582
Std reward : 0.493


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- **Ambiente:** con `is_slippery=True`, `env.step(action)` deja de ser determinístico — la acción elegida solo se ejecuta con cierta probabilidad, y el resto de las veces el agente se desliza hacia una dirección perpendicular no deseada (equivalente al concepto de "piso resbaloso" que vimos en el laboratorio del robot de almacén con `T(s,a,s')`). Esto se refleja en el desempeño final: con `is_slippery=False` la política entrenada logra `Mean reward = 1.000` con `Std = 0.000` (llega siempre a la meta), mientras que con `is_slippery=True`, incluso entrenando el doble de episodios (20,000), el resultado fue `Mean reward = 0.582` con `Std = 0.493` — el agente ya no puede garantizar el éxito porque el propio ambiente introduce incertidumbre que ninguna política puede eliminar del todo.
- **Algoritmo:** absolutamente nada cambia en la ecuación de actualización de Q-Learning. Sigue siendo exactamente `Q(s,a) += α[r + γ·max_a' Q(s',a') - Q(s,a)]`, la misma fórmula, el mismo código (`train_q_learning` no se modificó entre los dos experimentos). Esto ilustra que Q-Learning es **model-free**: nunca necesitó conocer `T(s,a,s')` explícitamente ni en la versión determinística ni en la estocástica — simplemente aprende de las muestras `(s,a,r,s')` que el ambiente le entrega en cada paso, sin importar qué tan predecible o ruidoso sea ese ambiente por dentro.


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> Almacena una estimación del **retorno total esperado y descontado** que el agente obtendría si, estando en el estado `s`, ejecuta la acción `a`, y de ahí en adelante sigue la política óptima. No es la recompensa inmediata: es una proyección de todo lo que vendría después, ponderado por `γ`.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> Sale de la propia Q-table del agente, evaluada en el estado siguiente `s'` al que se llegó tras ejecutar la acción. Representa la mejor estimación actual de "qué tan bueno es el mejor futuro posible desde `s'`" — se usa para construir el TD target `r + γ·max_a' Q(s',a')`, que es hacia donde `Q(s,a)` se va actualizando en cada paso.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Porque si el agente siempre actuara de forma puramente greedy desde el inicio (con `Q` en ceros o casi sin información), se quedaría repitiendo las primeras acciones que parecieron buenas por azar, sin explorar el resto del ambiente. `ε`-greedy balancea **exploración** (probar acciones al azar para descubrir información nueva) con **explotación** (usar lo ya aprendido). En este notebook `ε` decae exponencialmente con los episodios (`max_epsilon=1.0` a `min_epsilon=0.05`), de forma que el agente explora mucho al principio y cada vez más confía en su Q-table conforme entrena.

**4. ¿Por qué Q-Learning es model-free?**

> Porque nunca necesita conocer explícitamente la función de transición `T(s,a,s')` ni la función de recompensa `R(s)` del ambiente — a diferencia de Value Iteration o Policy Iteration, que sí requerían ese modelo completo para calcular `Σ_s' T(s,a,s')V(s')`. Q-Learning solo necesita **muestras de experiencia** `(s,a,r,s')` obtenidas al interactuar con el ambiente paso a paso, y actualiza `Q(s,a)` directamente a partir de esas muestras mediante el TD error, sin construir ni depender de un modelo explícito del mundo.
